> **Prefer Google Colab?** [Open this notebook in Colab](https://colab.research.google.com/github/SpatialTurn/DataAnalysis/blob/main/notebooks/01c_Travel_Time_to_Stores.ipynb).
> In Colab, click **File > Save a copy in Drive** first so your changes are kept.

# Travel Time to Grocery Stores

**Series:** Network Analysis (notebook 3 of 3)
**Time:** about 30 minutes

**In this notebook you will:**
- Download grocery store locations from OpenStreetMap
- Find the network center of a city
- Route from every store to the center and record distance and time
- Make a summary table and map, the first step of a food access study

**You should already know:** notebooks 01a and 01b, and notebook 2b of the Data Collection course's Geocoding module (OSM tags).

> **Before you start:** run each cell in order by clicking it and pressing **Shift + Enter**.

## The question

How far, by road, is each grocery store from the center of town?
The food desert case study uses the same idea at a larger scale: distance from where people live
to the nearest store.

## Step 1: Install, import and load the network

In [ ]:
# osmnx comes installed in the SpatialTurn GIS kernel, so there is nothing to install.

import osmnx as ox
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
place = "West Lafayette, Indiana, USA"
G = ox.graph_from_place(place, network_type="drive")
G = ox.add_edge_speeds(G, fallback=40)
G = ox.add_edge_travel_times(G)

## Step 2: Download grocery stores

We search for two tags, supermarkets and department stores. The search area is a little larger
than the road network, so some stores may sit just outside it. That is fine: they snap to the
closest road.

In [ ]:
stores = ox.features_from_place(place, tags={"shop": ["supermarket", "department_store"]})
stores = stores.reset_index()[["name", "shop", "geometry"]]
stores["geometry"] = stores.geometry.representative_point()
stores["name"] = stores["name"].fillna("Unnamed store")
stores

## Step 3: Find the center of town

We take the average longitude and latitude of all intersections, then snap it to the
nearest node. This is a simple stand in for "downtown".

In [ ]:
nodes, edges = ox.graph_to_gdfs(G)
center_lon, center_lat = nodes["x"].mean(), nodes["y"].mean()
center_node = ox.nearest_nodes(G, X=center_lon, Y=center_lat)
print("Center node:", center_node)

## Step 4: Route from each store to the center

The loop does the same three steps for every store: snap, route, measure.

In [ ]:
rows = []
routes = []
for _, store in stores.iterrows():
    store_node = ox.nearest_nodes(G, X=store.geometry.x, Y=store.geometry.y)
    route = ox.shortest_path(G, store_node, center_node, weight="travel_time")
    if route is None:          # no road connection found
        continue
    route_edges = ox.routing.route_to_gdf(G, route)
    routes.append(route)
    rows.append({
        "store": store["name"],
        "distance_km": round(route_edges["length"].sum() / 1000, 2),
        "drive_minutes": round(route_edges["travel_time"].sum() / 60, 1),
    })

results = pd.DataFrame(rows).sort_values("drive_minutes")
results

## Step 5: Map all the routes

In [ ]:
fig, ax = ox.plot_graph_routes(G, routes, route_linewidth=3, node_size=0,
                               bgcolor="white", show=False, close=False)
ax.scatter(center_lon, center_lat, c="red", s=80, zorder=5, label="Center")
ax.legend()
plt.show()

## Step 6: Classify access

Food access research often uses about **1 mile (1.6 km) in cities** and **10 miles (16 km) in
rural areas** as the cutoff for "low access". Here we label each store using the urban cutoff.

In [ ]:
results["within_1_mile"] = results["distance_km"] <= 1.6
results

In a real food desert study you would flip the question around: start from every
**neighborhood** (census tract) and find the distance to the **nearest** store.
That full workflow is in the Food Desert case study.

## Try it yourself

Repeat the analysis for **pharmacies** (`amenity=pharmacy`). Which pharmacy is the fastest drive from the center?

*Hint:* copy Steps 2 and 4 and change the tag.

<details>
<summary><b>Show a solution</b></summary>

```python
pharm = ox.features_from_place(place, tags={"amenity": "pharmacy"})
pharm = pharm.reset_index()[["name", "geometry"]]
pharm["geometry"] = pharm.geometry.representative_point()

rows = []
for _, p in pharm.iterrows():
    n = ox.nearest_nodes(G, X=p.geometry.x, Y=p.geometry.y)
    r = ox.shortest_path(G, n, center_node, weight="travel_time")
    if r:
        e = ox.routing.route_to_gdf(G, r)
        rows.append({"pharmacy": p["name"], "minutes": round(e["travel_time"].sum() / 60, 1)})
pd.DataFrame(rows).sort_values("minutes").head(1)
```

</details>

In [ ]:
# Write your answer here

## Recap

- OSM tags can take a list of values, such as `['supermarket', 'department_store']`.
- A loop of snap, route, measure gives distance and time for many places at once.
- Distance cutoffs (1 mile urban, 10 miles rural) turn distances into access categories.
- The food desert case study builds on this exact workflow.